# Лабораторная работа 1

Gavrishok Nikolai

Нужно предсказать, окажется ли число реакций на пост ниже обычного уровня автора, примерно таким же или выше. Используются текст публикации и предыдущие посты автора.

## Данные и запуск

В папке данных нужны `authors/`, `posts/`, `labels.csv` и `split.json`. Изображения и видео не используются. Библиотеки устанавливаются командой `python -m pip install -r submissions/gavrishok-nikolai/requirements.txt`.

Тетрадь запускается из корня репозитория. Укажите свой путь в `DATA`. Чтобы повторить обучение, измените `RUN_TRAINING` на `True`. При значении `False` выводятся сохранённые результаты.

In [1]:
from pathlib import Path
import json
import pandas as pd
from solution_gavrishok_nikolai import main

DATA = Path('data')
OUT = Path('results/gavrishok-nikolai')
SAVED = Path('submissions/gavrishok-nikolai')
RUN_TRAINING = False

## Метод

Официальное разделение по авторам сохранено. Внутри обучающей части 20% авторов отложены для сравнения вариантов. Одна половина этой группы используется для уточнения весов, вторая — для подтверждения результата.

Проверяются более точные веса смеси и дополнительные признаки: длина поста относительно обычной у автора, необычность перерыва перед публикацией и количество собственных постов за 7 и 30 дней. Выбран вариант с дополнительными признаками. Числовая модель даёт 80% смеси, текстовая — 20%; веса классов равны 0.8, 1.0, 0.8.

Реакции предсказываемого поста в признаки не входят. Код расчёта и обучения находится в `solution_gavrishok_nikolai.py`.

In [2]:
if RUN_TRAINING:
    main(['--data', str(DATA), '--out', str(OUT), '--final'])
else:
    print('Обучение пропущено. Ниже показаны сохранённые результаты.')

Обучение пропущено. Ниже показаны сохранённые результаты.


## Результаты

Основной показатель — macro-F1, среднее качество по трём классам с одинаковым весом каждого. Дополнительно приведены взвешенная каппа и доля верных ответов.

In [3]:
result_dir = OUT if RUN_TRAINING else SAVED
report = json.loads((result_dir / 'report.json').read_text(encoding='utf-8'))
scores = report['final_test']
print(f"macro-F1: {scores['macro_f1']:.4f}")
print(f"Взвешенная каппа: {scores['qwk']:.4f}")
print(f"Доля верных ответов: {scores['accuracy']:.4f}")
predictions = pd.read_csv(result_dir / 'predictions.csv')
print(f'Предсказаний: {len(predictions)}')
assert list(predictions.columns) == ['author', 'post', 'label']
assert len(predictions) == 17623
assert not predictions.duplicated(['author', 'post']).any()
assert predictions['label'].isin([0, 1, 2]).all()

macro-F1: 0.6144
Взвешенная каппа: 0.5385
Доля верных ответов: 0.6467
Предсказаний: 17623


## Вывод

Итоговый macro-F1 составил **0.6144** против **0.6005** у предыдущей версии. Новые признаки дали больший выигрыш, чем уточнение весов. Исходный пример в том же окружении получил **0.5403**. Вариант выбирался по внутренней проверке, после чего модель обучалась на всех обучающих авторах.

Число подписчиков и возраст публикации относятся к моменту сбора, как в исходном примере. Поэтому результат на этом наборе не равнозначен прогнозу для будущих публикаций.